# 01 - Raw Data Acquisition

Main purpose: Pull all the raw data necessary for the later steps. 

In [39]:
import pandas as pd
import numpy as np
import yfinance as yf
import os
from fredapi import Fred

In [40]:
# SPX returns
spx_raw = yf.download('^GSPC', start = '1996-01-01', multi_level_index = False).reset_index()
spx_raw['Returns'] = spx_raw['Close'].pct_change()
spx_raw = spx_raw.dropna()
spx_raw.to_csv('data/raw/01_spx_returns.csv', index=False)
spx_raw

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume,Returns
1,1996-01-03,621.320007,623.250000,619.559998,620.729980,468950000,0.000951
2,1996-01-04,617.700012,624.489990,613.960022,621.320007,512580000,-0.005826
3,1996-01-05,616.710022,617.700012,612.020020,617.700012,437110000,-0.001603
4,1996-01-08,618.460022,618.460022,616.489990,616.710022,130360000,0.002838
5,1996-01-09,609.450012,619.150024,608.210022,618.460022,417400000,-0.014568
...,...,...,...,...,...,...,...
7652,2026-06-01,7599.959961,7617.660156,7562.609863,7582.290039,6583960000,0.002625
7653,2026-06-02,7609.779785,7620.899902,7582.990234,7595.399902,5904410000,0.001292
7654,2026-06-03,7553.680176,7605.350098,7551.220215,7605.310059,5753860000,-0.007372
7655,2026-06-04,7584.310059,7598.189941,7516.540039,7516.540039,5398910000,0.004055


In [41]:
FRED_API_KEY = '0a07242543300b5582000574843752d6'
fred = Fred(api_key=FRED_API_KEY)

In [42]:
# FRED data
series = {
    #credit data
    'BAA': 'DBAA',
    'AAA': 'DAAA',
    'HY_OAS': 'BAMLH0A0HYM2',
    'IG_OAS': 'BAMLC0A0CM',
    
    #funding/financial conditions
    'TED': 'TEDRATE',
    'SOFR': 'SOFR',

    # Section F - Treasury yields
    'DGS3MO': 'DGS3MO',
    'DGS2': 'DGS2',
    'DGS10': 'DGS10',
    'DTB3': 'DTB3'
}

frames = []
for ID in series.values():
    frame = fred.get_series(ID, observation_start = '1996-01-01')
    frames.append(frame)

fred_raw = pd.concat(frames, axis = 1).reset_index()
fred_raw.columns = ['Date'] + list(series.keys())
fred_raw = fred_raw.ffill()
fred_raw.to_csv('data/raw/02_fred_raw.csv', index=False)
fred_raw

/var/folders/3f/c9b7x0c96rd39cz0kp8cc_nr0000gn/T/ipykernel_8288/3596419158.py:25: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  fred_raw = pd.concat(frames, axis = 1).reset_index()


,Date,BAA,AAA,HY_OAS,IG_OAS,TED,SOFR,DGS3MO,DGS2,DGS10,DTB3
0,1996-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1996-01-02,7.38,6.72,NaN,NaN,0.66,NaN,5.20,5.18,5.60,4.97
2,1996-01-03,7.37,6.71,NaN,NaN,0.58,NaN,5.20,5.17,5.58,5.05
3,1996-01-04,7.44,6.78,NaN,NaN,0.57,NaN,5.19,5.17,5.65,5.04
4,1996-01-05,7.50,6.80,NaN,NaN,0.58,NaN,5.19,5.20,5.69,5.03
...,...,...,...,...,...,...,...,...,...,...,...
7946,2026-05-31,6.02,5.50,2.74,0.74,0.09,3.63,3.69,3.98,4.45,3.60
7947,2026-06-01,6.01,5.49,2.72,0.73,0.09,3.65,3.78,4.05,4.47,3.63
7948,2026-06-02,6.00,5.48,2.71,0.74,0.09,3.63,3.77,4.05,4.46,3.63
7949,2026-06-03,6.03,5.51,2.75,0.74,0.09,3.61,3.78,4.08,4.49,3.63


In [43]:
#yfinance data

series = {
    'HYG': 'HYG',
    'LQD': 'LQD',
    'XLF': 'XLF',
    'WTI': 'CL=F',
    'USD': 'DX-Y.NYB',
    'GOLD': 'GC=F'
}

frames = []
for ID in series.values():
    frame = yf.download(ID, start = '1996-01-01', multi_level_index = False)['Close']
    frames.append(frame)

yf_raw = pd.concat(frames, axis = 1).reset_index()
yf_raw.columns = ['Date'] + list(series.keys())
yf_raw['WTI']  = yf_raw['WTI'].ffill()
yf_raw['GOLD'] = yf_raw['GOLD'].ffill()
yf_raw['USD']  = yf_raw['USD'].ffill()
yf_raw.to_csv('data/raw/03_yf_raw.csv', index=False)
yf_raw
    

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
/var/folders/3f/c9b7x0c96rd39cz0kp8cc_nr0000gn/T/ipykernel_8288/3951330954.py:17: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  yf_raw = pd.concat(frames, axis = 1).reset_index()


,Date,HYG,LQD,XLF,WTI,USD,GOLD
0,1996-01-01,NaN,NaN,NaN,NaN,84.760002,NaN
1,1996-01-02,NaN,NaN,NaN,NaN,84.690002,NaN
2,1996-01-03,NaN,NaN,NaN,NaN,85.110001,NaN
3,1996-01-04,NaN,NaN,NaN,NaN,85.220001,NaN
4,1996-01-05,NaN,NaN,NaN,NaN,85.059998,NaN
...,...,...,...,...,...,...,...
7719,2026-06-02,79.900002,108.919998,51.459999,93.760002,99.220001,4489.100098
7720,2026-06-03,79.680000,108.620003,50.869999,96.019997,99.529999,4436.700195
7721,2026-06-04,79.830002,108.849998,52.189999,93.040001,99.410004,4475.799805
7722,2026-06-05,79.430000,108.169998,52.299999,90.540001,100.070000,4337.100098
